# FMG additional analyses — Jupyter runner

Run this separate notebook from the FMG repository root. It calls the new scripts in `analysis/`; it does not alter the original notebook, manuscript, or archived results. Set data paths below. Primary analyses require the exact versions in `run_manifest_20261025.json`. The public-health S7 check currently stops on an unresolved saved-versus-regenerated metric mismatch.

In [ ]:
from pathlib import Path
import json, subprocess, sys

# Start Jupyter from the FMG repository root, or replace with Path(r'C:\\full\\path\\to\\FMG-research-git')
REPO_ROOT = Path.cwd()
ANALYSIS = REPO_ROOT / 'analysis'
assert (ANALYSIS / 'fmg_decomposition.py').is_file(), f'Not at repository root: {REPO_ROOT}'
print('Repository:', REPO_ROOT)
print('Kernel Python:', sys.executable)

## 1. Set the raw data paths

Use Windows paths in local Jupyter or `/content/...` paths in Colab. The files are not automatically uploaded by this notebook.

In [ ]:
ADULT = None       # r'C:\data\adult.zip'
COMPAS = None      # r'C:\data\compas-scores-two-years.csv'
TAIWAN = None      # r'C:\data\UCI_Credit_Card.csv'
GERMAN_OPENML = False  # optional exploratory dataset
BRFSS = None       # official 2024 XPT or ZIP
UCI_DIABETES = None  # r'C:\data\diabetes+130-us-hospitals.zip'

def checked(value, label):
    if value is None: return None
    p = Path(value).expanduser()
    if not p.is_file(): raise FileNotFoundError(f'{label} not found: {p}')
    return str(p.resolve())

ADULT, COMPAS, TAIWAN = checked(ADULT,'Adult'), checked(COMPAS,'COMPAS'), checked(TAIWAN,'Taiwan')
BRFSS, UCI_DIABETES = checked(BRFSS,'BRFSS'), checked(UCI_DIABETES,'UCI Diabetes')
print('Primary paths set:', all([ADULT, COMPAS, TAIWAN]))
print('Extension paths:', {'BRFSS': BRFSS, 'UCI': UCI_DIABETES})

## 2. Check the environment and script options

This only reports versions; it does not install or upgrade packages.

In [ ]:
manifest = json.loads((REPO_ROOT / 'run_manifest_20261025.json').read_text(encoding='utf-8'))
pins = (manifest.get('environment') or manifest.get('runtime'))['packages']
from importlib.metadata import version, PackageNotFoundError
installed = {}
for name in pins:
    try: installed[name] = version(name)
    except PackageNotFoundError: installed[name] = 'not-installed'
print('Package differences (required vs installed):', {n:(v,installed[n]) for n,v in pins.items() if installed[n] != v} or 'none')
for name in ['fmg_decomposition.py','threshold_sensitivity.py','extend_public_health_s7.py','audit_checklist.py']:
    print('\n---', name, '--help ---')
    subprocess.run([sys.executable, str(ANALYSIS / name), '--help'], cwd=REPO_ROOT, check=True)

## 3. Primary decomposition

Run only after setting all three primary data paths and using the recorded package pins. It validates against archived results and stops rather than forcing a match.

In [ ]:
if not all([ADULT, COMPAS, TAIWAN]):
    print('Skipped: set ADULT, COMPAS, and TAIWAN paths above.')
else:
    cmd = [sys.executable, str(ANALYSIS/'fmg_decomposition.py'), '--adult', ADULT, '--compas', COMPAS, '--taiwan', TAIWAN]
    if GERMAN_OPENML: cmd.append('--german-openml')
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

## 4. Threshold sensitivity

Primary runs use the primary pins. Extension runs use the separate pins in the public-health manifest; use a separate runtime for those data. Thresholds are selected from clean training data only.

In [ ]:
if all([ADULT, COMPAS, TAIWAN]):
    cmd = [sys.executable, str(ANALYSIS/'threshold_sensitivity.py'), '--adult', ADULT, '--compas', COMPAS, '--taiwan', TAIWAN]
    if GERMAN_OPENML: cmd.append('--german-openml')
elif BRFSS or UCI_DIABETES:
    cmd = [sys.executable, str(ANALYSIS/'threshold_sensitivity.py'), '--extension-only']
    if BRFSS: cmd += ['--brfss', BRFSS]
    if UCI_DIABETES: cmd += ['--diabetes-zip', UCI_DIABETES]
else:
    cmd = None
if cmd: subprocess.run(cmd, cwd=REPO_ROOT, check=True)
else: print('Skipped: set primary or extension paths above.')

## 5. Public-health S7-style table

The add-on verifies metrics against the saved run before writing. At the last check, UCI/LR/MCAR/5% differed (regenerated FMG 0.000000; archived 0.000302), so this cell is expected to stop until the source discrepancy is resolved.

In [ ]:
if not (BRFSS or UCI_DIABETES):
    print('Skipped: set BRFSS and/or UCI_DIABETES path above.')
else:
    cmd = [sys.executable, str(ANALYSIS/'extend_public_health_s7.py')]
    if BRFSS: cmd += ['--brfss', BRFSS]
    if UCI_DIABETES: cmd += ['--diabetes-zip', UCI_DIABETES]
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

## 6. Tests and reporting checklist

This runs the new tests plus the existing public-health primitive tests. The checklist runs only when a new per-seed CSV exists.

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(ANALYSIS/'tests'), str(REPO_ROOT/'experiments'/'public_health'/'test_audit_primitives.py')], cwd=REPO_ROOT, check=True)
for csv in [ANALYSIS/'outputs'/'fmg_decomposition_per_seed.csv', ANALYSIS/'outputs'/'public_health'/'public_health_s7_style_new.csv']:
    if csv.is_file():
        print('\nChecklist:', csv)
        subprocess.run([sys.executable, str(ANALYSIS/'audit_checklist.py'), str(csv)], cwd=REPO_ROOT, check=True)
print('Outputs folder:', ANALYSIS/'outputs')